# Sentiment Analysis using NLP & Machine Learning

## Objective
The objective of this project is to build a sentiment analysis model using NLP preprocessing and machine learning algorithms to classify text as positive or negative.

In [1]:
# Basic Libraries
import pandas as pd
import numpy as np

# NLP Libraries
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [2]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rutuj\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rutuj\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rutuj\AppData\Roaming\nltk_data...


True

In [3]:
!pip install datasets

     -------------------------------------- 515.2/515.2 kB 3.0 MB/s eta 0:00:00
     -------------------------------------- 625.2/625.2 kB 2.8 MB/s eta 0:00:00
     ---------------------------------------- 78.4/78.4 kB 4.3 MB/s eta 0:00:00
     ---------------------------------------- 73.5/73.5 kB 4.0 MB/s eta 0:00:00
     ---------------------------------------- 26.2/26.2 MB 2.2 MB/s eta 0:00:00
     -------------------------------------- 133.5/133.5 kB 4.0 MB/s eta 0:00:00
     -------------------------------------- 201.0/201.0 kB 2.4 MB/s eta 0:00:00
     ---------------------------------------- 64.7/64.7 kB 1.8 MB/s eta 0:00:00
     -------------------------------------- 464.0/464.0 kB 4.1 MB/s eta 0:00:00
     ---------------------------------------- 78.8/78.8 kB 4.3 MB/s eta 0:00:00
     ---------------------------------------- 56.8/56.8 kB 1.5 MB/s eta 0:00:00
     ---------------------------------------- 3.7/3.7 MB 2.9 MB/s eta 0:00:00
     -------------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 5.2.2 requires pyqt5<5.13, which is not installed.
spyder 5.2.2 requires pyqtwebengine<5.13, which is not installed.
conda-repo-cli 1.0.24 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.24 requires nbformat==5.4.0, but you have nbformat 5.7.0 which is incompatible.
conda-repo-cli 1.0.24 requires requests==2.28.1, but you have requests 2.32.5 which is incompatible.


In [4]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("imdb")

train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

# Combine both
df = pd.concat([train_df, test_df], ignore_index=True)

# Rename columns
df.columns = ['review', 'label']

# Convert label to text
df['sentiment'] = df['label'].map({1: 'positive', 0: 'negative'})

df.head()

README.md: 0.00B [00:00, ?B/s]

C:\Users\rutuj\anaconda3\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rutuj\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

,review,label,sentiment
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,negative
1,"""I Am Curious: Yellow"" is a risible and preten...",0,negative
2,If only to avoid making this type of film in t...,0,negative
3,This film was probably inspired by Godard's Ma...,0,negative
4,"Oh, brother...after hearing about this ridicul...",0,negative


In [5]:
print("Shape:", df.shape)
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

Shape: (50000, 3)

Class Distribution:
negative    25000
positive    25000
Name: sentiment, dtype: int64


## Data Understanding

The dataset contains 50,000 movie reviews.
It is balanced with equal number of positive and negative reviews.

In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()  # lowercase
    
    text = re.sub(r'<.*?>', '', text)  # remove HTML
    text = re.sub(r'http\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation
    
    words = word_tokenize(text)  # tokenize
    
    words = [w for w in words if w not in stop_words]  # remove stopwords
    
    words = [lemmatizer.lemmatize(w) for w in words]  # lemmatization
    
    return " ".join(words)

In [7]:
print("Cleaning text... (this will take 1–2 minutes)")

df['clean_review'] = df['review'].apply(preprocess)

print("Done!")
df[['review', 'clean_review']].head()

Cleaning text... (this will take 1–2 minutes)


LookupError: 
**********************************************************************
  Resource [93momw-1.4[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('omw-1.4')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/omw-1.4[0m

  Searched in:
    - 'C:\\Users\\rutuj/nltk_data'
    - 'C:\\Users\\rutuj\\anaconda3\\nltk_data'
    - 'C:\\Users\\rutuj\\anaconda3\\share\\nltk_data'
    - 'C:\\Users\\rutuj\\anaconda3\\lib\\nltk_data'
    - 'C:\\Users\\rutuj\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


In [8]:
import nltk
nltk.download('omw-1.4')

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\rutuj\AppData\Roaming\nltk_data...


True

In [9]:
df['clean_review'] = df['review'].apply(preprocess)

## Text Preprocessing

The text data was cleaned using the following steps:
- Converted text to lowercase
- Removed HTML tags and URLs
- Removed punctuation and special characters
- Tokenized text into words
- Removed stopwords
- Applied lemmatization

This helps in reducing noise and improving model performance.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_review']).toarray()
y = df['label']

In [11]:
print("Feature shape:", X.shape)

Feature shape: (50000, 5000)


## Feature Engineering using TF-IDF

TF-IDF (Term Frequency - Inverse Document Frequency) is used to convert text into numerical features.

It gives importance to words that are frequent in a document but rare across all documents, improving model performance.

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [13]:
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 40000
Testing samples: 10000


## Train-Test Split

The dataset is divided into training and testing sets.

- 80% data is used for training the model
- 20% data is used for testing the model

This helps in evaluating model performance on unseen data.

# 1: Logistic Regression

In [14]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

# 2: Naive Bayes

In [15]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

# 3: Decision Tree

In [16]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

## Model Building

Three machine learning models were used:

- Logistic Regression: Works well with text data
- Naive Bayes: Fast and effective for NLP tasks
- Decision Tree: Used for comparison

Each model was trained on the training dataset and predictions were made on the test dataset.

In [17]:
def evaluate(y_test, y_pred):
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

In [18]:
print("🔹 Logistic Regression")
evaluate(y_test, y_pred_lr)

print("\n🔹 Naive Bayes")
evaluate(y_test, y_pred_nb)

print("\n🔹 Decision Tree")
evaluate(y_test, y_pred_dt)

🔹 Logistic Regression
Accuracy : 0.8841
Precision: 0.8727845608507286
Recall   : 0.8962588473205257
F1 Score : 0.8843659582959192

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.87      0.88      5055
           1       0.87      0.90      0.88      4945

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


🔹 Naive Bayes
Accuracy : 0.8582
Precision: 0.8524885068958625
Recall   : 0.8624873609706775
F1 Score : 0.8574587856855651

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.85      0.86      5055
           1       0.85      0.86      0.86      4945

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000


🔹 Decision Tree
Accuracy : 0.713
Precision

## Model Evaluation

The models were evaluated using the following metrics:

- Accuracy
- Precision
- Recall
- F1 Score

F1 Score is considered the most important metric as it balances both precision and recall.

The performance of all models was compared to identify the best model.